In [9]:
# Importing necessary libraries
import pandas as pd
import sqlite3

conn = sqlite3.connect(":memory:")

def run(query): return pd.read_sql(query, conn)

In [10]:
# Loading the Excel file into pandas
df = pd.read_excel(r"C:\Projects\coffee-shop-analysis\data\Coffee Shop Sales.xlsx")
df.head(10)


,transaction_id,transaction_date,transaction_time,transaction_qty,store_id,store_location,product_id,unit_price,product_category,product_type,product_detail
0,1,2023-01-01,07:06:11,2,5,Lower Manhattan,32,3.00,Coffee,Gourmet brewed coffee,Ethiopia Rg
1,2,2023-01-01,07:08:56,2,5,Lower Manhattan,57,3.10,Tea,Brewed Chai tea,Spicy Eye Opener Chai Lg
2,3,2023-01-01,07:14:04,2,5,Lower Manhattan,59,4.50,Drinking Chocolate,Hot chocolate,Dark chocolate Lg
3,4,2023-01-01,07:20:24,1,5,Lower Manhattan,22,2.00,Coffee,Drip coffee,Our Old Time Diner Blend Sm
4,5,2023-01-01,07:22:41,2,5,Lower Manhattan,57,3.10,Tea,Brewed Chai tea,Spicy Eye Opener Chai Lg
5,6,2023-01-01,07:22:41,1,5,Lower Manhattan,77,3.00,Bakery,Scone,Oatmeal Scone
6,7,2023-01-01,07:25:49,1,5,Lower Manhattan,22,2.00,Coffee,Drip coffee,Our Old Time Diner Blend Sm
7,8,2023-01-01,07:33:34,2,5,Lower Manhattan,28,2.00,Coffee,Gourmet brewed coffee,Columbian Medium Roast Sm
8,9,2023-01-01,07:39:13,1,5,Lower Manhattan,39,4.25,Coffee,Barista Espresso,Latte Rg
9,10,2023-01-01,07:39:34,2,5,Lower Manhattan,58,3.50,Drinking Chocolate,Hot chocolate,Dark chocolate Rg


In [11]:
# Creating a new column "revenue" in DataFrame
df['revenue'] = df['transaction_qty']*df['unit_price']
print(df[['transaction_qty','unit_price','revenue']].head())

   transaction_qty  unit_price  revenue
0                2         3.0      6.0
1                2         3.1      6.2
2                2         4.5      9.0
3                1         2.0      2.0
4                2         3.1      6.2


In [12]:
# Loading DataFrame in SQL
df.to_sql("coffee", conn , index=False, if_exists="replace")

run("SELECT * FROM coffee LIMIT 10")

,transaction_id,transaction_date,transaction_time,transaction_qty,store_id,store_location,product_id,unit_price,product_category,product_type,product_detail,revenue
0,1,2023-01-01 00:00:00,07:06:11.000000,2,5,Lower Manhattan,32,3.00,Coffee,Gourmet brewed coffee,Ethiopia Rg,6.00
1,2,2023-01-01 00:00:00,07:08:56.000000,2,5,Lower Manhattan,57,3.10,Tea,Brewed Chai tea,Spicy Eye Opener Chai Lg,6.20
2,3,2023-01-01 00:00:00,07:14:04.000000,2,5,Lower Manhattan,59,4.50,Drinking Chocolate,Hot chocolate,Dark chocolate Lg,9.00
3,4,2023-01-01 00:00:00,07:20:24.000000,1,5,Lower Manhattan,22,2.00,Coffee,Drip coffee,Our Old Time Diner Blend Sm,2.00
4,5,2023-01-01 00:00:00,07:22:41.000000,2,5,Lower Manhattan,57,3.10,Tea,Brewed Chai tea,Spicy Eye Opener Chai Lg,6.20
...,...,...,...,...,...,...,...,...,...,...,...,...
149111,149452,2023-06-30 00:00:00,20:18:41.000000,2,8,Hell's Kitchen,44,2.50,Tea,Brewed herbal tea,Peppermint Rg,5.00
149112,149453,2023-06-30 00:00:00,20:25:10.000000,2,8,Hell's Kitchen,49,3.00,Tea,Brewed Black tea,English Breakfast Lg,6.00
149113,149454,2023-06-30 00:00:00,20:31:34.000000,1,8,Hell's Kitchen,45,3.00,Tea,Brewed herbal tea,Peppermint Lg,3.00
149114,149455,2023-06-30 00:00:00,20:57:19.000000,1,8,Hell's Kitchen,40,3.75,Coffee,Barista Espresso,Cappuccino,3.75


In [13]:
# Understanding the table scheme through PRAGMA
run("PRAGMA table_info(coffee)")

,cid,name,type,notnull,dflt_value,pk
0,0,transaction_id,INTEGER,0,None,0
1,1,transaction_date,TIMESTAMP,0,None,0
2,2,transaction_time,TIME,0,None,0
3,3,transaction_qty,INTEGER,0,None,0
4,4,store_id,INTEGER,0,None,0
5,5,store_location,TEXT,0,None,0
6,6,product_id,INTEGER,0,None,0
7,7,unit_price,REAL,0,None,0
8,8,product_category,TEXT,0,None,0
9,9,product_type,TEXT,0,None,0


In [14]:
# Find top 3 product types per store location by revenue using a window function

run("""SELECT * FROM 
(SELECT store_location,
product_type, SUM(revenue) AS revenue ,
RANK() OVER(PARTITION BY 
store_location ORDER BY SUM(revenue) DESC) 
AS Store_rank FROM coffee GROUP BY store_location, product_type) 
WHERE Store_rank <= 3""")

,store_location,product_type,revenue,Store_rank
0,Astoria,Barista Espresso,27935.00,1
1,Astoria,Brewed Chai tea,27427.90,2
2,Astoria,Hot chocolate,26335.25,3
3,Hell's Kitchen,Barista Espresso,32420.20,1
4,Hell's Kitchen,Brewed Chai tea,25645.30,2
5,Hell's Kitchen,Hot chocolate,23586.25,3
6,Lower Manhattan,Barista Espresso,31051.00,1
7,Lower Manhattan,Brewed Chai tea,24008.75,2
8,Lower Manhattan,Gourmet brewed coffee,23201.20,3


In [15]:
result = run("""SELECT * FROM 
(SELECT store_location,
product_type, SUM(revenue) AS revenue ,
RANK() OVER(PARTITION BY 
store_location ORDER BY SUM(revenue) DESC) 
AS Store_rank FROM coffee GROUP BY store_location, product_type) 
WHERE Store_rank <= 3""")

print(len(result))

9
